# Pandas de A à Z sur le dataset Beobank

**Support de formation — manipulation de données bancaires réelles avec Pandas**
Formation Beobank · Orsys

Ce notebook a deux objectifs :

1. **Partie 1 — Cours complet Pandas (A à Z)** : chaque notion de Pandas est expliquée
   simplement et appliquée directement sur les données de la banque (comptes, clients,
   adresses, opérations). Le code est **commenté ligne par ligne**, comme pour un débutant.
2. **Partie 2 — 100 cas d'utilisation bancaires** : des questions métier concrètes
   ("combien de clients ?", "quels comptes sont en découvert ?", "quelles sont les
   opérations les plus fréquentes ?"...) résolues avec Pandas.
3. **Partie 3 — Fiche récapitulative** : toutes les fonctions Pandas utilisées, classées et
   expliquées en une phrase.

## Le dataset

5 fichiers CSV, exports bancaires réels (anonymisés), séparateur `;`, encodage `cp1252` :

| Fichier | Contenu | Clé |
|---|---|---|
| `CTR.csv` | Comptes / contrats bancaires | `IDT_AC` (identifiant compte) |
| `TIE.csv` | Tiers = clients (personnes physiques ou morales) | `NUM_TIE` |
| `TIE_ADR.csv` | Coordonnées et adresses des clients | `NUM_TIE` |
| `TIE_X_CTR.csv` | Table de liaison : quel client détient quel compte | `NUM_TIE` + `IDT_AC` |
| `TXN_X_CTR.csv` | Opérations (mouvements) sur les comptes | `IDT_AC` |

**Schéma relationnel :**

```
TIE (clients) ──< TIE_X_CTR >── CTR (comptes) ──< TXN_X_CTR (opérations)
   │
   └── TIE_ADR (adresses, 1 pour 1 avec TIE)
```

> ⚠️ Ce dataset est un extrait pédagogique (quelques centaines de lignes). Les résultats
> chiffrés servent à illustrer la méthode, pas à produire une analyse métier définitive.

## Sommaire

**Partie 1 — Cours Pandas A à Z**
[A](#a) Import · [B](#b) Chargement CSV · [C](#c) Exploration · [D](#d) Colonnes/index ·
[E](#e) Sélection colonnes · [F](#f) Sélection lignes · [G](#g) Filtrage booléen ·
[H](#h) Valeurs manquantes · [I](#i) Types & conversions · [J](#j) Tri ·
[K](#k) Nouvelles colonnes · [L](#l) Texte (`.str`) · [M](#m) Dates (`.dt`) ·
[N](#n) Doublons · [O](#o) Comptages · [P](#p) GroupBy · [Q](#q) Tableaux croisés ·
[R](#r) Fusion (merge/concat) · [S](#s) Réorganisation (melt/pivot) · [T](#t) Index avancé ·
[U](#u) Cumuls & fenêtres · [V](#v) Export · [W](#w) Copies vs vues · [X](#x) Vectorisation ·
[Y](#y) Pipeline complet · [Z](#z) Pièges fréquents

**Partie 2 — [100 cas d'utilisation bancaires](#part2)**

**Partie 3 — [Fiche récapitulative des fonctions](#part3)**

<a id="a"></a>
## A. Importer les bibliothèques

Toujours la même convention dans tout projet Python : on importe `pandas` sous l'alias `pd`
et `numpy` sous l'alias `np`.

In [1]:
import pandas as pd   # pd : alias standard de Pandas (manipulation de tableaux de données)
import numpy as np    # np : alias standard de NumPy (calcul numérique, valeurs manquantes NaN)

print("Pandas version :", pd.__version__)   # .__version__ : affiche la version installée
print("NumPy version  :", np.__version__)

Pandas version : 3.0.1
NumPy version  : 2.4.3


<a id="b"></a>
## B. Charger les fichiers CSV

Les fichiers bancaires utilisent :
- le séparateur `;` (et non `,`) → paramètre `sep=";"`
- l'encodage Windows `cp1252` (à cause des accents français/néerlandais) → paramètre `encoding="cp1252"`

On écrit une **fonction** réutilisable pour ne pas répéter ces paramètres 5 fois.

In [2]:
DATA_DIR = "../data/"   # dossier contenant les 5 fichiers CSV (chemin relatif au notebook)

def charger(nom_fichier):
    """Charge un export CSV bancaire (sep ';' , encodage cp1252) et renvoie un DataFrame."""
    chemin = DATA_DIR + nom_fichier          # concatène dossier + nom de fichier
    df = pd.read_csv(chemin, sep=";", encoding="utf-8")   # lit le CSV dans un DataFrame
    return df

# On charge les 5 tables. Chaque DataFrame porte un nom court, réutilisé dans tout le notebook.
ctr = charger("CTR.csv")          # ctr : comptes / contrats
tie = charger("TIE.csv")          # tie : clients (tiers)
adr = charger("TIE_ADR.csv")      # adr : adresses des clients
lnk = charger("TIE_X_CTR.csv")    # lnk : liaison client <-> compte (link)
txn = charger("TXN_X_CTR.csv")    # txn : opérations sur les comptes

print("Comptes (ctr)      :", ctr.shape)   # .shape : (nb lignes, nb colonnes)
print("Clients (tie)       :", tie.shape)
print("Adresses (adr)      :", adr.shape)
print("Liaisons (lnk)      :", lnk.shape)
print("Opérations (txn)    :", txn.shape)

Comptes (ctr)      : (200, 11)
Clients (tie)       : (100, 11)
Adresses (adr)      : (100, 20)
Liaisons (lnk)      : (200, 6)
Opérations (txn)    : (1260, 10)


<a id="c"></a>
## C. Première exploration d'un DataFrame

Avant toute analyse, on regarde toujours : à quoi ressemblent les données, combien de
lignes/colonnes, quels types.

In [3]:
ctr.head()   # .head() : affiche les 5 premières lignes (par défaut) -> aperçu rapide

,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
0,65500004701,29862201102,2024-05-29,6,2025-12-10,2025-12-10,EUR,.,.,.,.
1,65500006391,29912218433,2024-08-07,6,2025-02-05,2025-02-05,EUR,.,.,.,.
2,65500007774,29922113324,2022-01-12,4,2022-01-12,.,EUR,.,.,.,.
3,65500008787,29872222935,2025-01-07,4,2025-01-07,.,EUR,.,2025-10-31,.,.
4,65500014230,29862237513,2025-01-18,6,2025-02-15,2025-02-15,EUR,.,.,.,.


In [4]:
ctr.tail(3)   # .tail(n) : affiche les n dernières lignes

,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
197,65500491523,25304103771,2026-01-09,4,2026-01-09,.,EUR,.,.,.,.
198,65500491571,25244098277,2026-01-09,4,2026-01-09,.,EUR,.,.,.,.
199,65500491575,25754099879,2026-01-09,4,2026-01-09,.,EUR,0.00,.,0.00,.


In [5]:
ctr.sample(5, random_state=0)   # .sample(n) : n lignes prises au hasard (random_state = résultat reproductible)

,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
18,65500050418,29282017114,2020-12-01,6,2024-08-01,2024-08-01,EUR,1219.68,2025-10-31,.,0.00
170,65500477836,25184076782,2025-11-28,4,2025-11-28,.,EUR,.,.,.,.
107,65500263412,29382295348,2024-02-05,4,2024-02-05,.,EUR,.,.,.,.
98,65500212213,90080819123,2015-02-26,6,2016-04-11,2016-04-11,EUR,.,.,.,.
177,65500490225,25104097985,2025-12-14,4,2025-12-14,.,EUR,.,.,.,.


In [6]:
ctr.info()   # .info() : nombre de lignes, liste des colonnes, type de chaque colonne, mémoire utilisée

<class 'pandas.DataFrame'>
RangeIndex: 200 entries, 0 to 199
Data columns (total 11 columns):
 #   Column       Non-Null Count  Dtype
---  ------       --------------  -----
 0   IDT_AC       200 non-null    int64
 1   REF_CTR_INN  200 non-null    int64
 2   DAT_OUV_CTR  200 non-null    str  
 3   COD_ECV_CTR  200 non-null    int64
 4   DAT_ECV_CTR  200 non-null    str  
 5   DAT_CLO_CTR  200 non-null    str  
 6   COD_DEV      200 non-null    str  
 7   SLD_CTR      200 non-null    str  
 8   DAT_MAJ_SLD  200 non-null    str  
 9   SLD_DSP      200 non-null    str  
 10  MNT_INI      200 non-null    str  
dtypes: int64(3), str(8)
memory usage: 17.3 KB


In [7]:
ctr.dtypes   # .dtypes : uniquement le type de chaque colonne (int64, object = texte, float64...)

IDT_AC         int64
REF_CTR_INN    int64
DAT_OUV_CTR      str
COD_ECV_CTR    int64
DAT_ECV_CTR      str
DAT_CLO_CTR      str
COD_DEV          str
SLD_CTR          str
DAT_MAJ_SLD      str
SLD_DSP          str
MNT_INI          str
dtype: object

In [8]:
ctr.describe()   # .describe() : statistiques (moyenne, min, max, quartiles) des colonnes numériques

,IDT_AC,REF_CTR_INN,COD_ECV_CTR
count,2.000000e+02,2.000000e+02,200.000000
mean,6.550026e+10,4.585308e+10,4.950000
std,1.603949e+05,2.769239e+10,1.001255
min,6.550000e+10,2.507410e+10,4.000000
25%,6.550012e+10,2.914974e+10,4.000000
50%,6.550023e+10,2.968213e+10,4.000000
75%,6.550042e+10,9.008092e+10,6.000000
max,6.550049e+10,9.009739e+10,6.000000


In [9]:
ctr.columns   # .columns : liste des noms de colonnes

Index(['IDT_AC', 'REF_CTR_INN', 'DAT_OUV_CTR', 'COD_ECV_CTR', 'DAT_ECV_CTR',
       'DAT_CLO_CTR', 'COD_DEV', 'SLD_CTR', 'DAT_MAJ_SLD', 'SLD_DSP',
       'MNT_INI'],
      dtype='str')

In [16]:
ctr.index   # .index : les étiquettes de ligne (ici un simple numéro de 0 à 199, l'index par défaut)

RangeIndex(start=0, stop=200, step=1)

<a id="d"></a>
## D. Renommer des colonnes

Les noms de colonnes bancaires sont cryptiques (héritage SAS/COBOL). On peut les renommer
pour rendre le code plus lisible, avec `.rename()`.

In [20]:
# .rename(columns={...}) : renvoie une COPIE avec les colonnes renommées (ne modifie pas ctr)
apercu = ctr.rename(columns={
    "IDT_AC": "id_compte",          # identifiant du compte
    "DAT_OUV_CTR": "date_ouverture",  # date d'ouverture du contrat
    "COD_DEV": "devise",              # code devise (EUR, USD...)
})
apercu.head(10)

,id_compte,REF_CTR_INN,date_ouverture,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,devise,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
0,65500004701,29862201102,2024-05-29,6,2025-12-10,2025-12-10,EUR,.,.,.,.
1,65500006391,29912218433,2024-08-07,6,2025-02-05,2025-02-05,EUR,.,.,.,.
2,65500007774,29922113324,2022-01-12,4,2022-01-12,.,EUR,.,.,.,.
3,65500008787,29872222935,2025-01-07,4,2025-01-07,.,EUR,.,2025-10-31,.,.
4,65500014230,29862237513,2025-01-18,6,2025-02-15,2025-02-15,EUR,.,.,.,.
5,65500016903,29892298341,2024-05-29,6,2025-12-10,2025-12-10,EUR,.,2025-10-31,.,.
6,65500017438,29882152002,2020-12-09,4,2020-12-09,.,USD,2.82,2025-02-12,2.82,.
7,65500018665,29242121728,2022-09-28,6,2024-11-27,2024-11-27,EUR,.,.,.,.
8,65500019025,29902113259,2022-02-22,6,2023-01-05,2023-01-05,EUR,.,.,.,.
9,65500021862,29222189161,2021-07-29,6,2024-07-06,2024-07-06,EUR,0.00,2025-10-31,.,0.00


<a id="e"></a>
## E. Sélectionner des colonnes

- `df["col"]` → une colonne, renvoyée comme une **Series** (1 dimension)
- `df[["col1", "col2"]]` → plusieurs colonnes, renvoyées comme un **DataFrame**

In [30]:
ctr["COD_ECV_CTR"]   # une seule colonne -> Series (une seule liste de valeurs indexées)

0      6
1      6
2      4
3      4
4      6
      ..
195    4
196    4
197    4
198    4
199    4
Name: COD_ECV_CTR, Length: 200, dtype: int64

In [28]:
ctr["REF_CTR_INN"]   # une seule colonne -> Series (une seule liste de valeurs indexées)

0      29862201102
1      29912218433
2      29922113324
3      29872222935
4      29862237513
          ...     
195    25184084526
196    25804103498
197    25304103771
198    25244098277
199    25754099879
Name: REF_CTR_INN, Length: 200, dtype: int64

In [31]:
ctr[["IDT_AC", "COD_DEV", "DAT_OUV_CTR"]]   # liste de colonnes -> DataFrame (sous-tableau)

,IDT_AC,COD_DEV,DAT_OUV_CTR
0,65500004701,EUR,2024-05-29
1,65500006391,EUR,2024-08-07
2,65500007774,EUR,2022-01-12
3,65500008787,EUR,2025-01-07
4,65500014230,EUR,2025-01-18
...,...,...,...
195,65500491455,EUR,2026-01-07
196,65500491460,EUR,2026-01-07
197,65500491523,EUR,2026-01-09
198,65500491571,EUR,2026-01-09


In [32]:
ctr.select_dtypes(include="number")   # select_dtypes : garde uniquement les colonnes d'un type donné (ici numériques)

,IDT_AC,REF_CTR_INN,COD_ECV_CTR
0,65500004701,29862201102,6
1,65500006391,29912218433,6
2,65500007774,29922113324,4
3,65500008787,29872222935,4
4,65500014230,29862237513,6
...,...,...,...
195,65500491455,25184084526,4
196,65500491460,25804103498,4
197,65500491523,25304103771,4
198,65500491571,25244098277,4


In [34]:
ctr.select_dtypes(include=["object", "string"])


,DAT_OUV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
0,2024-05-29,2025-12-10,2025-12-10,EUR,.,.,.,.
1,2024-08-07,2025-02-05,2025-02-05,EUR,.,.,.,.
2,2022-01-12,2022-01-12,.,EUR,.,.,.,.
3,2025-01-07,2025-01-07,.,EUR,.,2025-10-31,.,.
4,2025-01-18,2025-02-15,2025-02-15,EUR,.,.,.,.
...,...,...,...,...,...,...,...,...
195,2026-01-07,2026-01-07,.,EUR,.,.,.,.
196,2026-01-07,2026-01-07,.,EUR,.,.,.,.
197,2026-01-09,2026-01-09,.,EUR,.,.,.,.
198,2026-01-09,2026-01-09,.,EUR,.,.,.,.


In [35]:
ctr.select_dtypes(exclude="number")


,DAT_OUV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
0,2024-05-29,2025-12-10,2025-12-10,EUR,.,.,.,.
1,2024-08-07,2025-02-05,2025-02-05,EUR,.,.,.,.
2,2022-01-12,2022-01-12,.,EUR,.,.,.,.
3,2025-01-07,2025-01-07,.,EUR,.,2025-10-31,.,.
4,2025-01-18,2025-02-15,2025-02-15,EUR,.,.,.,.
...,...,...,...,...,...,...,...,...
195,2026-01-07,2026-01-07,.,EUR,.,.,.,.
196,2026-01-07,2026-01-07,.,EUR,.,.,.,.
197,2026-01-09,2026-01-09,.,EUR,.,.,.,.
198,2026-01-09,2026-01-09,.,EUR,.,.,.,.


<a id="f"></a>
## F. Sélectionner des lignes : `.loc` et `.iloc`

- `.loc[étiquette]` : sélection par **étiquette** (nom d'index / nom de colonne)
- `.iloc[position]` : sélection par **position numérique** (comme une liste Python)

In [51]:
ctr.loc[0]  # .loc[0:3] : les lignes avec les index 0, 1, 2, 3 (inclusif pour .loc)

IDT_AC         65500004701
REF_CTR_INN    29862201102
DAT_OUV_CTR     2024-05-29
COD_ECV_CTR              6
DAT_ECV_CTR     2025-12-10
DAT_CLO_CTR     2025-12-10
COD_DEV                EUR
SLD_CTR                  .
DAT_MAJ_SLD              .
SLD_DSP                  .
MNT_INI                  .
Name: 0, dtype: object

In [48]:
ctr.iloc[0:3]       # .iloc[0:3] : les lignes en position 0, 1, 2 (comme un slice Python classique)

,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
0,65500004701,29862201102,2024-05-29,6,2025-12-10,2025-12-10,EUR,.,.,.,.
1,65500006391,29912218433,2024-08-07,6,2025-02-05,2025-02-05,EUR,.,.,.,.
2,65500007774,29922113324,2022-01-12,4,2022-01-12,.,EUR,.,.,.,.


In [59]:
ctr.loc[0, "COD_DEV"]        # .loc[ligne, colonne] : valeur à l'étiquette de ligne 0, colonne "COD_DEV"

'EUR'

In [55]:
ctr.loc[0:2, ["IDT_AC", "COD_DEV"]]   # .loc avec un slice ET une liste de colonnes

,IDT_AC,COD_DEV
0,65500004701,EUR
1,65500006391,EUR
2,65500007774,EUR


<a id="g"></a>
## G. Filtrer des lignes (filtrage booléen)

C'est **la** compétence la plus utilisée en Pandas : on écrit une condition, Pandas renvoie
une Series de `True`/`False`, qu'on utilise ensuite pour filtrer le DataFrame.

Opérateurs à retenir : `&` (et), `|` (ou), `~` (non) — **toujours entre parenthèses**.

In [60]:
masque = ctr["COD_DEV"] == "EUR"    # masque : Series de True/False, une valeur par ligne de ctr
masque.head()

0    True
1    True
2    True
3    True
4    True
Name: COD_DEV, dtype: bool

In [62]:
comptes_eur = ctr[ctr["COD_DEV"] == "EUR"]   # on filtre ctr avec le masque booléen -> uniquement les comptes en EUR
print(comptes_eur.shape)


(193, 11)


In [63]:
comptes_eur.head(3)

,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
0,65500004701,29862201102,2024-05-29,6,2025-12-10,2025-12-10,EUR,.,.,.,.
1,65500006391,29912218433,2024-08-07,6,2025-02-05,2025-02-05,EUR,.,.,.,.
2,65500007774,29922113324,2022-01-12,4,2022-01-12,.,EUR,.,.,.,.


In [77]:
#    - Le code de l'état du compte (COD_ECV_CTR) doit être égal à 4 (comptes actifs)
#    - La devise du compte (COD_DEV) doit être égale à "EUR" (en euros)
comptes_actifs_eur = ctr[(ctr["COD_ECV_CTR"] == 4) & (ctr["COD_DEV"] == "EUR")]
# 2. Affichage des dimensions du tableau filtré sous forme de tuple : (nombre_de_lignes, nombre_de_colonnes)
print(comptes_actifs_eur.shape)
comptes_actifs_eur.head(3)


(104, 11)


,IDT_AC,REF_CTR_INN,DAT_OUV_CTR,COD_ECV_CTR,DAT_ECV_CTR,DAT_CLO_CTR,COD_DEV,SLD_CTR,DAT_MAJ_SLD,SLD_DSP,MNT_INI
2,65500007774,29922113324,2022-01-12,4,2022-01-12,.,EUR,.,.,.,.
3,65500008787,29872222935,2025-01-07,4,2025-01-07,.,EUR,.,2025-10-31,.,.
13,65500033145,29252120420,2022-08-23,4,2022-08-23,.,EUR,361.74,2026-04-01,361.74,.


In [72]:
# 1. Filtrage avec la méthode .isin() : on extrait les lignes du DataFrame 'ctr'
#    dont la devise (COD_DEV) correspond exactement à l'une des valeurs de la liste donnée
comptes_etrangers = ctr[ctr["COD_DEV"].isin(["USD", "AUD", "NOK"])]

# 2. Extraction et affichage d'un sous-ensemble de colonnes :
#    on utilise une double paire de crochets `[[...]]` pour ne conserver et n'afficher 
#    que les colonnes de l'identifiant ("IDT_AC") et de la devise ("COD_DEV")
comptes_etrangers[["IDT_AC", "COD_DEV"]]
print(colonnes_selectionnees.shape[1])
colonnes_selectionnees.head(3)

2


,IDT_AC,COD_DEV
6,65500017438,USD
10,65500022341,AUD
103,65500248622,NOK


In [78]:
# .query() : même filtrage, écrit comme une phrase (pratique pour des conditions lisibles)
ctr.query("COD_DEV == 'EUR' and COD_ECV_CTR == 4").shape

(104, 11)

<a id="h"></a>
## H. Valeurs manquantes

**Piège du dataset bancaire** : les valeurs manquantes ne sont pas de vraies cellules vides,
mais la chaîne de caractères `"."` (convention de l'export SAS). Il faut donc d'abord les
convertir en vraies valeurs manquantes Pandas (`NA`) avant de pouvoir les détecter ou les
traiter avec `.isna()`.

In [80]:
# Avant nettoyage : "." est un texte comme un autre, Pandas ne le reconnaît pas comme manquant
(ctr["SLD_CTR"] == ".").sum()   # nombre de lignes où le solde est encore le texte "."

np.int64(137)

In [81]:
# .replace(".", pd.NA) : remplace le texte "." par une vraie valeur manquante, sur tout le DataFrame
ctr = ctr.replace(".", pd.NA)
tie = tie.replace(".", pd.NA)
adr = adr.replace(".", pd.NA)
lnk = lnk.replace(".", pd.NA)
txn = txn.replace(".", pd.NA)

ctr["SLD_CTR"].isna().sum()   # .isna() : True si valeur manquante -> .sum() compte les True

np.int64(137)

In [82]:
ctr.isna().sum()   # nombre de valeurs manquantes, colonne par colonne

IDT_AC           0
REF_CTR_INN      0
DAT_OUV_CTR      0
COD_ECV_CTR      0
DAT_ECV_CTR      0
DAT_CLO_CTR    105
COD_DEV          0
SLD_CTR        137
DAT_MAJ_SLD    128
SLD_DSP        153
MNT_INI        160
dtype: int64

In [ ]:
# ==============================================================================
# 1. ÉTAT INITIAL : Comptage des valeurs renseignées (non manquantes) à l'origine
# ==============================================================================

# .notna() renvoie True pour chaque ligne remplie. 
# .sum() additionne ces True (qui valent 1), ce qui donne le nombre exact de sexes connus au départ.
nb_sexes_initiaux = tie["COD_SEX"].notna().sum()
print(f"Nombre de lignes avec 'COD_SEX' renseigné au départ : {nb_sexes_initiaux}")


# ==============================================================================
# 2. NETTOYAGE : Suppression des lignes où 'COD_SEX' est manquant
# ==============================================================================

# On crée le nouveau DataFrame nettoyé? SUPPRIMER les lignes où 'COD_SEX' est manquant
tie_avec_sexe = tie.dropna(subset=["COD_SEX"])


# ==============================================================================
# 3. ÉTAT FINAL : Validation et vérification de la cohérence du nettoyage
# ==============================================================================

# On compte à nouveau les valeurs non manquantes dans le tableau final, le nouveau tableau nétoyé
nb_sexes_finals = tie_avec_sexe["COD_SEX"].notna().sum()
print(f"Nombre de lignes avec 'COD_SEX' renseigné après nettoyage : {nb_sexes_finals}")

# Comparaison des dimensions globales des DataFrames ou tableau (lignes, colonnes)
print(f"Évolution de la taille du tableau : {tie.shape} -> {tie_avec_sexe.shape}")

# Vérification ultime : le nombre de lignes du nouveau tableau DOIT être strictement 
# égal au nombre de valeurs valides comptées au tout début.
if len(tie_avec_sexe) == nb_sexes_initiaux:
    print(" Validation réussie : Le tableau nettoyé ne contient plus aucune valeur manquante dans 'COD_SEX'.")
else:
    print("Attention : Il y a une anomalie dans le filtrage.")


Nombre de lignes avec 'COD_SEX' renseigné au départ : 99
Nombre de lignes avec 'COD_SEX' renseigné après nettoyage : 99
Évolution de la taille du tableau : (100, 11) -> (99, 11)
 Validation réussie : Le tableau nettoyé ne contient plus aucune valeur manquante dans 'COD_SEX'.


In [ ]:
# 1. Nettoyage des données manquantes (NaN) :
#    #on ligne les lignes où 'COD_SEX' est manquant avec subset= ["COD_SEX"]
#    Toutes les lignes où le sexe est vide/inconnu (NaN) sont définitivement supprimées dans le nouveau tableau 'tie_avec_sexe'.
#   cette ligne a créée une couvelle copie du DataFrame original 'tie' sans les valeurs manquantes dans 'COD_SEX'
tie_avec_sexe = tie.dropna(subset=["COD_SEX"])

# 2. Comparaison des dimensions avant et après nettoyage :
#    On affiche la taille d'origine (lignes, colonnes) suivie de la nouvelle taille.
#    Cela permet de voir visuellement combien de lignes ont été supprimées à cause d'une valeur manquante.
print(tie.shape, "->", tie_avec_sexe.shape)


(100, 11) -> (99, 11)


In [ ]:
# .fillna(valeur) : remplace les valeurs manquantes par une valeur de repli
tie["COD_SEX_rempli"] = tie["COD_SEX"].fillna("Non renseigné")
tie["COD_SEX_rempli"].value_counts()

COD_SEX_rempli
M                50
F                49
Non renseigné     1
Name: count, dtype: int64

<a id="i"></a>
## I. Convertir les types (nombres, dates)

À la lecture du CSV, **tout est du texte** dès qu'une colonne contient au moins une valeur
non numérique (comme `"."` avant nettoyage). Il faut convertir explicitement :

- `pd.to_numeric()` pour les nombres (montants, soldes)
- `pd.to_datetime()` pour les dates

In [89]:
# to_numeric(..., errors="coerce") : convertit en nombre, et met NaN si conversion impossible
ctr["SLD_CTR"] = pd.to_numeric(ctr["SLD_CTR"], errors="coerce")
ctr["SLD_DSP"] = pd.to_numeric(ctr["SLD_DSP"], errors="coerce")
ctr["MNT_INI"] = pd.to_numeric(ctr["MNT_INI"], errors="coerce")

ctr[["SLD_CTR", "SLD_DSP", "MNT_INI"]].dtypes   # les 3 colonnes sont maintenant numériques (float64)

SLD_CTR    float64
SLD_DSP    float64
MNT_INI    float64
dtype: object

In [ ]:
# to_datetime(..., errors="coerce") : convertit en date, la plupart des dates du fichier sont au format ISO (AAAA-MM-JJ)
for colonne in ["DAT_OUV_CTR", "DAT_ECV_CTR", "DAT_CLO_CTR", "DAT_MAJ_SLD"]:
    ctr[colonne] = pd.to_datetime(ctr[colonne], format="%Y-%m-%d", errors="coerce") #remplacer par une valeur manquante sans modifier le programme si la conversion échoue

for colonne in ["DAT_STA_FED", "DAT_PRE_CTR", "DAT_DER_CTR", "DAT_NAI"]:
    tie[colonne] = pd.to_datetime(tie[colonne], format="%Y-%m-%d", errors="coerce")

for colonne in ["DAT_CRE_MVT_CPB"]:
    txn[colonne] = pd.to_datetime(txn[colonne], format="%Y-%m-%d", errors="coerce")

ctr["DAT_OUV_CTR"].head()

0   2024-05-29
1   2024-08-07
2   2022-01-12
3   2025-01-07
4   2025-01-18
Name: DAT_OUV_CTR, dtype: datetime64[us]

## Formats de dates courants avec `pd.to_datetime()`

| Exemple de date | Format Python / Pandas | Signification |
|---|---|---|
| `2026-09-17` | `%Y-%m-%d` | AAAA-MM-JJ |
| `17/09/2026` | `%d/%m/%Y` | JJ/MM/AAAA |
| `09/17/2026` | `%m/%d/%Y` | MM/JJ/AAAA |
| `17-09-2026` | `%d-%m-%Y` | JJ-MM-AAAA |
| `2026/09/17` | `%Y/%m/%d` | AAAA/MM/JJ |
| `17.09.2026` | `%d.%m.%Y` | JJ.MM.AAAA |
| `20260917` | `%Y%m%d` | AAAAMMJJ |
| `17092026` | `%d%m%Y` | JJMMAAAA |
| `17/09/26` | `%d/%m/%y` | JJ/MM/AA |
| `26-09-17` | `%y-%m-%d` | AA-MM-JJ |
| `17 Sep 2026` | `%d %b %Y` | Jour + mois abrégé + année |
| `17 September 2026` | `%d %B %Y` | Jour + mois complet + année |
| `Sep 17, 2026` | `%b %d, %Y` | Mois abrégé + jour + année |
| `September 17, 2026` | `%B %d, %Y` | Mois complet + jour + année |
| `2026-09-17 14:30:00` | `%Y-%m-%d %H:%M:%S` | Date + heure |
| `17/09/2026 14:30` | `%d/%m/%Y %H:%M` | Date française + heure |
| `17-09-2026 14:30:45` | `%d-%m-%Y %H:%M:%S` | Date + heure complète |
| `2026-09-17T14:30:00` | `%Y-%m-%dT%H:%M:%S` | Format ISO avec heure |

### Signification des codes

| Code | Signification | Exemple |
|---|---|---|
| `%Y` | Année sur 4 chiffres | `2026` |
| `%y` | Année sur 2 chiffres | `26` |
| `%m` | Mois numérique | `09` |
| `%d` | Jour du mois | `17` |
| `%H` | Heure sur 24 h | `14` |
| `%M` | Minutes | `30` |
| `%S` | Secondes | `45` |
| `%b` | Mois abrégé | `Sep` |
| `%B` | Mois complet | `September` |

### Exemple avec Pandas

```python
df["date"] = pd.to_datetime(
    df["date"],
    format="%d/%m/%Y",
    errors="coerce"
)

In [92]:
# Piège classique : DEUX colonnes de dates de la MÊME table peuvent avoir des formats différents.
# Dans TIE_ADR et dans la colonne DAT_DCS de TIE, les dates sont écrites "24NOV2025"
# (format jour-MOIS-année abrégé) et non au format ISO -> il faut préciser le format
# explicitement avec le code %d%b%Y (d = jour, %b = mois abrégé en lettres, Y = année).
adr["DAT_MAJ_ADR"] = pd.to_datetime(adr["DAT_MAJ_ADR"], format="%d%b%Y", errors="coerce")
tie["DAT_DCS"] = pd.to_datetime(tie["DAT_DCS"], format="%d%b%Y", errors="coerce")
adr["DAT_MAJ_ADR"].head()

0   2025-11-24
1   2025-11-24
2   2026-01-09
3   2025-11-25
4   2025-11-26
Name: DAT_MAJ_ADR, dtype: datetime64[us]

In [94]:
# ── Traduire les codes métier en libellés plus lisibles ──────────────────────

# .map() permet de remplacer les valeurs d'une colonne à partir d'un dictionnaire.
# Le dictionnaire est écrit sous la forme :
# {code_original: "libellé souhaité"}

# Dans COD_ECV_CTR :
# 4 signifie que le compte est actif
# 6 signifie que le compte est clôturé
# On crée une nouvelle colonne STATUT_COMPTE sans modifier la colonne d'origine.
ctr["STATUT_COMPTE"] = ctr["COD_ECV_CTR"].map({
    4: "Actif",
    6: "Clôturé"
})

# Dans COD_TYP_TIE :
# 1 correspond à une personne physique
# 2 correspond à une personne morale
# On crée une nouvelle colonne TYPE_CLIENT avec des valeurs plus compréhensibles.
tie["TYPE_CLIENT"] = tie["COD_TYP_TIE"].map({
    1: "Personne physique",
    2: "Personne morale"
})

# value_counts() compte le nombre d'occurrences de chaque valeur
# présente dans la colonne STATUT_COMPTE.
# Exemple de résultat :
# Actif      850
# Clôturé    150
#
# Cela permet d'obtenir rapidement la répartition des comptes selon leur statut.
ctr["STATUT_COMPTE"].value_counts()

STATUT_COMPTE
Actif      105
Clôturé     95
Name: count, dtype: int64

In [96]:
# Création des nouvelles colonnes lisibles
ctr["STATUT_COMPTE"] = ctr["COD_ECV_CTR"].map({
    4: "Actif",
    6: "Clôturé"
})

tie["TYPE_CLIENT"] = tie["COD_TYP_TIE"].map({
    1: "Personne physique",
    2: "Personne morale"
})

# Afficher les premières lignes de la table ctr
# avec le code original et sa traduction
display(
    ctr[["COD_ECV_CTR", "STATUT_COMPTE"]].head(3)
)

# Afficher les premières lignes de la table tie
# avec le code original et sa traduction
display(
    tie[["COD_TYP_TIE", "TYPE_CLIENT"]].head(3)
)

,COD_ECV_CTR,STATUT_COMPTE
0,6,Clôturé
1,6,Clôturé
2,4,Actif


,COD_TYP_TIE,TYPE_CLIENT
0,2,Personne morale
1,1,Personne physique
2,1,Personne physique


<a id="j"></a>
## J. Trier les données

In [97]:
# .sort_values("colonne") : trie du plus petit au plus grand (ascending=True par défaut)
ctr.sort_values("DAT_OUV_CTR").head(3)[["IDT_AC", "DAT_OUV_CTR"]]

,IDT_AC,DAT_OUV_CTR
130,65500344419,2009-08-28
118,65500295573,2009-08-28
48,65500115203,2009-09-02


<a id="k"></a>
## K. Créer de nouvelles colonnes

Règle d'or Pandas : **on n'écrit quasiment jamais de boucle `for`** sur les lignes. On écrit
des opérations "vectorisées" qui s'appliquent à toute la colonne d'un coup (beaucoup plus
rapide et plus lisible).

In [110]:
# Opération vectorisée directe : différence entre solde comptable et solde disponible
ctr["ECART_SOLDE"] = ctr["SLD_CTR"] - ctr["SLD_DSP"]
ctr[["IDT_AC", "SLD_CTR", "SLD_DSP", "ECART_SOLDE"]].head(2)
print(ctr[["ECART_SOLDE"]].head(2))

   ECART_SOLDE
0          NaN
1          NaN


In [107]:
# np.where(condition, valeur_si_vrai, valeur_si_faux) : équivalent vectorisé d'un SI/ALORS/SINON
ctr["EN_DECOUVERT"] = np.where(ctr["SLD_CTR"] < 0, "Oui", "Non")
ctr["EN_DECOUVERT"].value_counts()

EN_DECOUVERT
Non    187
Oui     13
Name: count, dtype: int64

In [114]:
# .apply(fonction) : applique une fonction à chaque valeur (plus lent que le vectorisé, à réserver
# aux cas où il n'existe pas d'opération Pandas/NumPy toute faite)
def categorie_solde(solde):
    if pd.isna(solde): #est une fonction essentielle pour détecter les valeurs manquantes (comme NaN, None ou NaT)
        return "Inconnu"
    elif solde < 0:
        return "Négatif"
    elif solde == 0:
        return "Nul"
    else:
        return "Positif"

ctr["CATEGORIE_SOLDE"] = ctr["SLD_CTR"].apply(categorie_solde) # applique la fonction categorie_solde à chaque valeur de la colonne SLD_CTR
ctr["CATEGORIE_SOLDE"].value_counts() # affiche le nombre de comptes dans chaque catégorie de solde


CATEGORIE_SOLDE
Inconnu    137
Nul         38
Négatif     13
Positif     12
Name: count, dtype: int64

In [115]:
# .assign() : crée une ou plusieurs colonnes sans modifier le DataFrame d'origine (renvoie une copie)
apercu = ctr.assign(SOLDE_EN_CENTIMES=lambda d: d["SLD_CTR"] * 100) # crée une nouvelle colonne SOLDE_EN_CENTIMES sans modifier le DataFrame d'origine avec une fonction lamda d  
apercu[["IDT_AC", "SLD_CTR", "SOLDE_EN_CENTIMES"]].head(3)

,IDT_AC,SLD_CTR,SOLDE_EN_CENTIMES
0,65500004701,NaN,NaN
1,65500006391,NaN,NaN
2,65500007774,NaN,NaN


<a id="l"></a>
## L. Manipuler du texte avec l'accesseur `.str`

Toutes les méthodes Python habituelles sur le texte (`.upper()`, `.strip()`, `.contains()`...)
existent en version "colonne entière" via `.str` : `colonne.str.methode()`.

In [116]:
txn["LIB_OPE_INL_1"].str.upper().head(3)   # .str.upper() : met en majuscules toute la colonne

0    REJ NATIONALE THA COMPTE SOLDE
1                DOMICILIATION POUR
2                   BEOBANK BELGIUM
Name: LIB_OPE_INL_1, dtype: str

In [119]:
# .str.contains("motif", case=False) : True si le texte contient le motif (case=False -> insensible à la casse)
virements = txn[txn["LIB_OPE_INL_1"].str.contains("virement", case=False, na=False)] # filtre les transactions contenant le mot "virement" (insensible à la casse)
virements[["IDT_AC", "LIB_OPE_INL_1"]].head(3) # affiche un aperçu des transactions contenant le mot "virement"

,IDT_AC,LIB_OPE_INL_1
5,65500003192,Virement de
6,65500003192,Virement de
10,65500003192,Virement de


In [120]:
# .str.split("@") : découpe le texte -> utile pour extraire le domaine d'un email
adr["DOMAINE_EMAIL"] = adr["ADR_EMA"].str.split("@").str[-1]   # .str[-1] : dernier élément de la liste découpée
adr[["ADR_EMA", "DOMAINE_EMAIL"]].dropna().head(3) # affiche un aperçu des adresses email avec leur domaine, en supprimant les valeurs manquantes

,ADR_EMA,DOMAINE_EMAIL
0,PRO@FREE.BE,FREE.BE
1,TTSAXL@E-I.COM,E-I.COM
2,BART.JANSSENS@GMAIL.COM,GMAIL.COM


<a id="m"></a>
## M. Manipuler des dates avec l'accesseur `.dt`

Une fois une colonne convertie en date (voir section I), `.dt` donne accès à l'année, au
mois, au jour de la semaine, et permet de calculer des durées.

In [121]:
ctr["ANNEE_OUVERTURE"] = ctr["DAT_OUV_CTR"].dt.year     # .dt.year : extrait l'année
ctr["MOIS_OUVERTURE"] = ctr["DAT_OUV_CTR"].dt.month     # .dt.month : extrait le mois (1 à 12)
ctr[["DAT_OUV_CTR", "ANNEE_OUVERTURE", "MOIS_OUVERTURE"]].head(3)

,DAT_OUV_CTR,ANNEE_OUVERTURE,MOIS_OUVERTURE
0,2024-05-29,2024,5
1,2024-08-07,2024,8
2,2022-01-12,2022,1


In [124]:
# Différence entre deux dates -> Timedelta, converti en jours avec .dt.days
aujourdhui = pd.Timestamp.today() # récupère la date et l'heure actuelles sous forme de Timestamp
ctr["ANCIENNETE_JOURS"] = (aujourdhui - ctr["DAT_OUV_CTR"]).dt.days # calcule l'ancienneté en jours
ctr[["DAT_OUV_CTR", "ANCIENNETE_JOURS"]].head(3) # affiche un aperçu de l'ancienneté des comptes en jours

,DAT_OUV_CTR,ANCIENNETE_JOURS
0,2024-05-29,841
1,2024-08-07,771
2,2022-01-12,1709


In [126]:
txn["JOUR_SEMAINE"] = txn["DAT_CRE_MVT_CPB"].dt.day_name()   # .dt.day_name() : nom du jour ("Monday", "Tuesday"...)
txn["JOUR_SEMAINE"].value_counts() # affiche le nombre de transactions par jour de la semaine

JOUR_SEMAINE
Monday       560
Friday       545
Wednesday     65
Tuesday       56
Thursday      32
Saturday       2
Name: count, dtype: int64

<a id="n"></a>
## N. Détecter et supprimer les doublons

In [127]:
ctr.duplicated().sum()   # .duplicated() : True si la ligne est identique à une ligne précédente -> .sum() les compte

np.int64(0)

In [128]:
ctr["IDT_AC"].duplicated().sum()   # même logique appliquée à une seule colonne -> identifiants en double

np.int64(0)

In [129]:
ctr_sans_doublons = ctr.drop_duplicates()   # .drop_duplicates() : supprime les lignes strictement identiques
print(ctr.shape, "->", ctr_sans_doublons.shape)

(200, 19) -> (200, 19)


<a id="o"></a>
## O. Comptages et valeurs uniques

In [130]:
tie["COD_LNG_CTR"].unique()   # .unique() : liste des valeurs distinctes (sans les compter)

<StringArray>
['FR', 'NL']
Length: 2, dtype: str

In [131]:
tie["COD_LNG_CTR"].nunique()   # .nunique() : NOMBRE de valeurs distinctes

2

In [132]:
tie["COD_LNG_CTR"].value_counts()   # .value_counts() : compte les occurrences de chaque valeur, triées du + fréquent au - fréquent

COD_LNG_CTR
FR    67
NL    33
Name: count, dtype: int64

In [133]:
tie["COD_LNG_CTR"].value_counts(normalize=True).round(3) * 100   # normalize=True : donne des proportions (%) au lieu de comptages bruts

COD_LNG_CTR
FR    67.0
NL    33.0
Name: proportion, dtype: float64

<a id="p"></a>
## P. Agréger avec `.groupby()`

`.groupby("colonne")` regroupe les lignes qui partagent la même valeur, puis on applique
un calcul (moyenne, somme, comptage...) sur chaque groupe. C'est l'équivalent Pandas du
`GROUP BY` en SQL.

In [134]:
# groupby + .size() : nombre de lignes par groupe
ctr.groupby("COD_DEV").size()

COD_DEV
AUD      1
EUR    193
NOK      1
USD      5
dtype: int64

In [135]:
# groupby + .agg({...}) : plusieurs statistiques différentes par colonne, en un seul appel
ctr.groupby("STATUT_COMPTE").agg(
    nb_comptes=("IDT_AC", "count"),     # count : nombre de valeurs non manquantes
    solde_moyen=("SLD_CTR", "mean"),    # mean : moyenne
    solde_total=("SLD_CTR", "sum"),     # sum : somme
)

,nb_comptes,solde_moyen,solde_total
STATUT_COMPTE,,,
Actif,105,-19597.562368,-744707.37
Clôturé,95,48.787200,1219.68


In [136]:
# .transform() : renvoie un résultat de même taille que le DataFrame d'origine (utile pour comparer
# chaque ligne à la moyenne de son groupe, sans réduire le nombre de lignes)
ctr["SOLDE_MOYEN_DEVISE"] = ctr.groupby("COD_DEV")["SLD_CTR"].transform("mean")
ctr[["IDT_AC", "COD_DEV", "SLD_CTR", "SOLDE_MOYEN_DEVISE"]].head(3)

,IDT_AC,COD_DEV,SLD_CTR,SOLDE_MOYEN_DEVISE
0,65500004701,EUR,NaN,-12391.5085
1,65500006391,EUR,NaN,-12391.5085
2,65500007774,EUR,NaN,-12391.5085


<a id="q"></a>
## Q. Tableaux croisés : `pivot_table` et `crosstab`

- `pd.crosstab(a, b)` : compte les combinaisons entre deux colonnes catégorielles
- `df.pivot_table(...)` : comme un tableau croisé dynamique Excel

In [137]:
pd.crosstab(tie["COD_LNG_CTR"], tie["TYPE_CLIENT"])   # nombre de clients par langue x type de client

TYPE_CLIENT,Personne morale,Personne physique
COD_LNG_CTR,,
FR,1,66
NL,0,33


In [ ]:
ctr.pivot_table(
    values="SLD_CTR",        # colonne à agréger
    index="STATUT_COMPTE",   # lignes du tableau
    columns="COD_DEV",       # colonnes du tableau
    aggfunc="mean",          # fonction d'agrégation
)

<a id="r"></a>
## R. Fusionner des tables : `merge` et `concat`

C'est **le cœur** de ce dataset relationnel : `merge` relie deux tables sur une clé commune
(comme une jointure SQL). `how="left"` conserve toutes les lignes de la table de gauche, même
sans correspondance à droite.

In [ ]:
# Jointure clients <-> liaisons : on ajoute à chaque client ses comptes
clients_comptes = tie.merge(lnk, on="NUM_TIE", how="left")
print(tie.shape, "+", lnk.shape, "->", clients_comptes.shape)
clients_comptes[["NUM_TIE", "TYPE_CLIENT", "IDT_AC"]].head(3)

In [ ]:
# Jointure à 3 tables en chaîne : clients -> liaisons -> comptes
vue_complete = (
    tie.merge(lnk, on="NUM_TIE", how="left")          # étape 1 : clients + liaisons
       .merge(ctr, on="IDT_AC", how="left")            # étape 2 : + informations des comptes
)
vue_complete[["NUM_TIE", "TYPE_CLIENT", "IDT_AC", "STATUT_COMPTE", "SLD_CTR"]].head(3)

In [ ]:
# how="outer" : garde TOUTES les lignes des deux côtés, y compris sans correspondance
# -> utile pour un contrôle de cohérence (voir cas d'utilisation n°73)
controle = ctr.merge(lnk, on="IDT_AC", how="outer", indicator=True)
controle["_merge"].value_counts()   # indicator=True ajoute une colonne "_merge" qui dit d'où vient chaque ligne

In [ ]:
# pd.concat([...]) : empile des DataFrames (contrairement à merge qui les fusionne côte à côte)
comptes_eur = ctr[ctr["COD_DEV"] == "EUR"].head(2)
comptes_usd = ctr[ctr["COD_DEV"] == "USD"].head(2)
pd.concat([comptes_eur, comptes_usd])[["IDT_AC", "COD_DEV"]]

<a id="s"></a>
## S. Réorganiser un tableau : `pivot` et `melt`

- `melt` : passe d'un format "large" à un format "long" (dépivote)
- `pivot` : l'opération inverse (repivote)

In [ ]:
# Les 4 colonnes de libellé d'une opération (LIB_OPE_INL_1 à 4) peuvent être "empilées" en une seule colonne
extrait = txn[["IDT_AC", "LIB_OPE_INL_1", "LIB_OPE_INL_2"]].head(3)
extrait_long = extrait.melt(id_vars="IDT_AC", var_name="partie_libelle", value_name="texte")
extrait_long

<a id="t"></a>
## T. Index avancé : `set_index` / `reset_index`

In [ ]:
# .set_index("colonne") : remplace l'index numérique par une colonne -> accès direct via .loc
ctr_par_compte = ctr.set_index("IDT_AC")
ctr_par_compte.loc[65500004701]   # accès direct à la ligne du compte n°65500004701, sans filtrage booléen

In [ ]:
# .reset_index() : l'opération inverse, remet un index numérique 0,1,2... et range l'ancien index en colonne
ctr_par_compte.reset_index().head(2)

<a id="u"></a>
## U. Cumuls et fenêtres glissantes

In [ ]:
# .cumsum() : somme cumulée -> utile pour visualiser une évolution dans le temps
operations_triees = txn.sort_values("DAT_CRE_MVT_CPB")
operations_triees["NB_OPE_CUMULE"] = range(1, len(operations_triees) + 1)
operations_triees[["DAT_CRE_MVT_CPB", "NB_OPE_CUMULE"]].head(3)

In [ ]:
# .rolling(n).mean() : moyenne glissante sur les n dernières valeurs -> lisse une série dans le temps
operations_par_mois = txn.groupby(txn["DAT_CRE_MVT_CPB"].dt.to_period("M")).size()
operations_par_mois.rolling(3).mean().tail(6)   # moyenne glissante sur 3 mois

<a id="v"></a>
## V. Exporter des résultats

In [ ]:
resume_comptes = ctr.groupby("STATUT_COMPTE").agg(nb_comptes=("IDT_AC", "count"), solde_moyen=("SLD_CTR", "mean"))

resume_comptes.to_csv("resume_comptes.csv", sep=";", encoding="utf-8")   # .to_csv() : écrit un fichier CSV
print("Export CSV terminé")

try:
    resume_comptes.to_excel("resume_comptes.xlsx")   # .to_excel() : nécessite le paquet 'openpyxl'
    print("Export Excel terminé")
except ImportError:
    print("Export Excel ignoré : le paquet 'openpyxl' n'est pas installé (pip install openpyxl)")

<a id="w"></a>
## W. Copies vs vues (le piège `SettingWithCopyWarning`)

Quand on filtre un DataFrame (`df[condition]`), le résultat est **parfois** une vue sur les
données d'origine, pas toujours une vraie copie indépendante. Si on modifie ensuite ce résultat,
Pandas peut afficher un avertissement. Règle simple : dès qu'on veut modifier un sous-ensemble
filtré, on l'écrit explicitement avec `.copy()`.

In [ ]:
# Bonne pratique : .copy() après un filtrage, avant modification
comptes_usd = ctr[ctr["COD_DEV"] == "USD"].copy()   # .copy() : copie indépendante, sûre à modifier
comptes_usd["COMMENTAIRE"] = "Compte en devise étrangère"   # aucune ambiguïté, aucun avertissement
comptes_usd[["IDT_AC", "COD_DEV", "COMMENTAIRE"]]

<a id="x"></a>
## X. Vectorisation vs boucle `for` (pourquoi Pandas est rapide)

Une opération "vectorisée" (`colonne_a - colonne_b`) est calculée en une fois, en langage bas
niveau (C), sur toute la colonne. Une boucle `for` ligne par ligne repasse par Python à chaque
itération : beaucoup plus lent. Sur un petit dataset la différence est invisible, mais sur des
millions de lignes bancaires, elle devient critique.

In [ ]:
import time

# Méthode lente : boucle for ligne par ligne
depart = time.perf_counter()
resultats = []
for valeur in ctr["SLD_CTR"]:
    resultats.append(valeur * 1.0 if pd.notna(valeur) else np.nan)
duree_boucle = time.perf_counter() - depart

# Méthode rapide : opération vectorisée
depart = time.perf_counter()
resultat_vectorise = ctr["SLD_CTR"] * 1.0
duree_vectorisee = time.perf_counter() - depart

print(f"Boucle for      : {duree_boucle*1000:.3f} ms")
print(f"Vectorisé       : {duree_vectorisee*1000:.3f} ms")

<a id="y"></a>
## Y. Résumé : un pipeline de nettoyage complet

On rassemble toutes les étapes précédentes (chargement, nettoyage, typage) dans une seule
fonction réutilisable — le réflexe à avoir en début de tout projet Pandas.

In [ ]:
def charger_et_nettoyer_comptes():
    """Pipeline complet : charge CTR.csv, nettoie les valeurs manquantes, type les colonnes."""
    df = charger("CTR.csv")                          # 1. chargement
    df = df.replace(".", pd.NA)                       # 2. valeurs manquantes
    for c in ["SLD_CTR", "SLD_DSP", "MNT_INI"]:        # 3. colonnes numériques
        df[c] = pd.to_numeric(df[c], errors="coerce")
    for c in ["DAT_OUV_CTR", "DAT_ECV_CTR", "DAT_CLO_CTR", "DAT_MAJ_SLD"]:   # 4. colonnes dates
        df[c] = pd.to_datetime(df[c], format="%Y-%m-%d", errors="coerce")
    df["STATUT_COMPTE"] = df["COD_ECV_CTR"].map({4: "Actif", 6: "Clôturé"})  # 5. libellés métier
    return df

comptes_propres = charger_et_nettoyer_comptes()
comptes_propres.info()

<a id="z"></a>
## Z. Pièges fréquents pour un débutant

1. **Confondre `df["a"]["b"]` et `df.loc[a, b]`** : le premier peut déclencher un avertissement
   de copie ("chained indexing"), le second est toujours sûr.
2. **Oublier `errors="coerce"`** sur `to_numeric`/`to_datetime` : une seule valeur invalide fait
   planter toute la conversion sans lui.
3. **Comparer une colonne encore en texte à un nombre** (`"100" > 50` lève une erreur ou donne
   un résultat faux) : toujours convertir le type d'abord.
4. **`inplace=True`** : évitez-le, il complique le débogage et sera progressivement retiré de
   Pandas ; préférez `df = df.methode(...)`.
5. **Oublier `.copy()`** après un filtrage quand on prévoit de modifier le résultat.
6. **`&`/`|` sans parenthèses** dans un filtre : `df[a == 1 & b == 2]` est faux, il faut
   `df[(a == 1) & (b == 2)]`.

<a id="part2"></a>
# Partie 2 — 100 cas d'utilisation bancaires

100 questions métier concrètes, résolues avec Pandas sur les données de la banque.
Regroupées en 10 catégories de 10 cas. On repart des DataFrames nettoyés de la Partie 1
(`ctr`, `tie`, `adr`, `lnk`, `txn`).

## Catégorie 1 — Portefeuille clients (`tie`)

### Cas 1 — Combien de clients au total dans le portefeuille ?

In [ ]:
nb_clients = tie.shape[0]
print(f"Nombre total de clients : {nb_clients}")

### Cas 2 — Quelle est la répartition personnes physiques / personnes morales ?

In [ ]:
tie["TYPE_CLIENT"].value_counts()

### Cas 3 — Quelle est la répartition hommes / femmes (avec les valeurs manquantes) ?

In [ ]:
tie["COD_SEX"].value_counts(dropna=False)

### Cas 4 — Quelle est la répartition des clients par langue de contact ?

In [ ]:
tie["COD_LNG_CTR"].value_counts()

### Cas 5 — Quel est l'âge de chaque client (personne physique) ?

In [ ]:
aujourdhui = pd.Timestamp.today()
tie["AGE"] = ((aujourdhui - tie["DAT_NAI"]).dt.days / 365.25).round(1)
tie[["NUM_TIE", "TYPE_CLIENT", "DAT_NAI", "AGE"]].dropna(subset=["AGE"]).head(5)

### Cas 6 — Comment se répartissent les clients par tranche d'âge ?

In [ ]:
bornes = [0, 25, 40, 60, 75, 120]
etiquettes = ["<25 ans", "25-39 ans", "40-59 ans", "60-74 ans", "75 ans et +"]
tie["TRANCHE_AGE"] = pd.cut(tie["AGE"], bins=bornes, labels=etiquettes)
tie["TRANCHE_AGE"].value_counts().sort_index()

### Cas 7 — Qui sont le client le plus âgé et le client le plus jeune ?

In [ ]:
tie_age = tie.dropna(subset=["AGE"])
plus_age = tie_age.loc[tie_age["AGE"].idxmax()]
plus_jeune = tie_age.loc[tie_age["AGE"].idxmin()]
print(f"Client le plus âgé   : NUM_TIE={plus_age['NUM_TIE']} -> {plus_age['AGE']} ans")
print(f"Client le plus jeune : NUM_TIE={plus_jeune['NUM_TIE']} -> {plus_jeune['AGE']} ans")

### Cas 8 — Combien de clients sont enregistrés comme décédés ?

In [ ]:
clients_decedes = tie[tie["DAT_DCS"].notna()]
print(f"Clients décédés enregistrés : {clients_decedes.shape[0]}")
clients_decedes[["NUM_TIE", "DAT_DCS"]]

### Cas 9 — Quelle est l'ancienneté (en années) du premier contrat de chaque client ?

In [ ]:
aujourdhui = pd.Timestamp.today()
tie["ANCIENNETE_RELATION"] = ((aujourdhui - tie["DAT_PRE_CTR"]).dt.days / 365.25).round(1)
tie[["NUM_TIE", "DAT_PRE_CTR", "ANCIENNETE_RELATION"]].dropna(subset=["ANCIENNETE_RELATION"]).sort_values("ANCIENNETE_RELATION", ascending=False).head(5)

### Cas 10 — Comment se répartit le statut fédéral des clients (`COD_STA_FED`) ?

In [ ]:
repartition = tie["COD_STA_FED"].value_counts().sort_index()
pourcentages = (tie["COD_STA_FED"].value_counts(normalize=True).sort_index() * 100).round(1)
pd.DataFrame({"nb_clients": repartition, "pourcentage": pourcentages})

## Catégorie 2 — Comptes / contrats (`ctr`)

### Cas 11 — Combien de comptes/contrats au total ?

In [ ]:
print(f"Nombre total de comptes : {ctr.shape[0]}")

### Cas 12 — Combien de comptes sont actifs vs clôturés ?

In [ ]:
ctr["STATUT_COMPTE"].value_counts()

### Cas 13 — Comment se répartissent les comptes par devise ?

In [ ]:
ctr["COD_DEV"].value_counts()

### Cas 14 — Quelle est l'ancienneté moyenne des comptes actifs ?

In [ ]:
aujourdhui = pd.Timestamp.today()
comptes_actifs = ctr[ctr["STATUT_COMPTE"] == "Actif"]
anciennete = (aujourdhui - comptes_actifs["DAT_OUV_CTR"]).dt.days / 365.25
print(f"Ancienneté moyenne des comptes actifs : {anciennete.mean():.1f} ans")

### Cas 15 — Combien de comptes ont été ouverts chaque année ?

In [ ]:
ctr["DAT_OUV_CTR"].dt.year.value_counts().sort_index()

### Cas 16 — Quelle a été la durée de vie des comptes clôturés ?

In [ ]:
comptes_clotures = ctr[ctr["STATUT_COMPTE"] == "Clôturé"].copy()
comptes_clotures["DUREE_VIE_JOURS"] = (comptes_clotures["DAT_CLO_CTR"] - comptes_clotures["DAT_OUV_CTR"]).dt.days
comptes_clotures[["IDT_AC", "DAT_OUV_CTR", "DAT_CLO_CTR", "DUREE_VIE_JOURS"]].sort_values("DUREE_VIE_JOURS").head(5)

### Cas 17 — Quels comptes ont été ouverts et clôturés la même année (rotation rapide) ?

In [ ]:
comptes_clotures = ctr[ctr["STATUT_COMPTE"] == "Clôturé"]
rotation_rapide = comptes_clotures[comptes_clotures["DAT_OUV_CTR"].dt.year == comptes_clotures["DAT_CLO_CTR"].dt.year]
print(f"Comptes ouverts/clôturés la même année : {rotation_rapide.shape[0]}")
rotation_rapide[["IDT_AC", "DAT_OUV_CTR", "DAT_CLO_CTR"]]

### Cas 18 — Quels comptes sont détenus en devise étrangère (hors EUR) ?

In [ ]:
comptes_etrangers = ctr[ctr["COD_DEV"] != "EUR"]
comptes_etrangers[["IDT_AC", "COD_DEV", "STATUT_COMPTE"]]

### Cas 19 — Quels sont les 5 comptes les plus anciens ?

In [ ]:
ctr.nsmallest(5, "DAT_OUV_CTR")[["IDT_AC", "DAT_OUV_CTR", "STATUT_COMPTE"]]

### Cas 20 — Quel est le taux de clôture global des comptes ?

In [ ]:
taux_cloture = (ctr["STATUT_COMPTE"] == "Clôturé").mean() * 100
print(f"Taux de clôture : {taux_cloture:.1f} %")

## Catégorie 3 — Soldes (`ctr`)

### Cas 21 — Combien de comptes ont un solde connu vs un solde manquant ?

In [ ]:
ctr["SLD_CTR"].isna().value_counts().rename({True: "Solde manquant", False: "Solde connu"})

### Cas 22 — Quel est le solde moyen et le solde médian des comptes actifs ?

In [ ]:
soldes_actifs = ctr.loc[ctr["STATUT_COMPTE"] == "Actif", "SLD_CTR"]
print(f"Solde moyen  : {soldes_actifs.mean():.2f} EUR")
print(f"Solde médian : {soldes_actifs.median():.2f} EUR")

### Cas 23 — Quels comptes sont en situation de découvert (solde négatif) ?

In [ ]:
comptes_decouvert = ctr[ctr["SLD_CTR"] < 0]
print(f"Comptes en découvert : {comptes_decouvert.shape[0]}")
comptes_decouvert[["IDT_AC", "SLD_CTR", "COD_DEV"]].sort_values("SLD_CTR")

### Cas 24 — Quel est le montant total en découvert sur le portefeuille ?

In [ ]:
total_decouvert = ctr.loc[ctr["SLD_CTR"] < 0, "SLD_CTR"].sum()
print(f"Total en découvert : {total_decouvert:.2f} EUR")

### Cas 25 — Quels sont les 10 plus gros soldes positifs ?

In [ ]:
ctr.nlargest(10, "SLD_CTR")[["IDT_AC", "SLD_CTR", "COD_DEV"]]

### Cas 26 — Quels sont les 10 plus gros découverts ?

In [ ]:
ctr.nsmallest(10, "SLD_CTR")[["IDT_AC", "SLD_CTR", "COD_DEV"]]

### Cas 27 — Comment se répartissent les comptes par tranche de solde ?

In [ ]:
bornes = [-float("inf"), -1000, 0, 1000, 10000, float("inf")]
etiquettes = ["Découvert important (< -1000)", "Découvert léger", "0 à 1000", "1000 à 10000", "Plus de 10000"]
ctr["TRANCHE_SOLDE"] = pd.cut(ctr["SLD_CTR"], bins=bornes, labels=etiquettes)
ctr["TRANCHE_SOLDE"].value_counts().sort_index()

### Cas 28 — Quel est l'écart entre solde comptable et solde disponible ?

In [ ]:
ctr["ECART_SOLDE"] = ctr["SLD_CTR"] - ctr["SLD_DSP"]
ctr[["IDT_AC", "SLD_CTR", "SLD_DSP", "ECART_SOLDE"]].dropna(subset=["ECART_SOLDE"]).sort_values("ECART_SOLDE", ascending=False).head(5)

### Cas 29 — Quels comptes ont un solde supérieur à leur montant initial (croissance) ?

In [ ]:
comptes_en_hausse = ctr[ctr["SLD_CTR"] > ctr["MNT_INI"]]
print(f"Comptes dont le solde a progressé depuis l'ouverture : {comptes_en_hausse.shape[0]}")
comptes_en_hausse[["IDT_AC", "MNT_INI", "SLD_CTR"]]

### Cas 30 — Quelle est la distribution statistique complète des soldes ?

In [ ]:
ctr["SLD_CTR"].describe()

## Catégorie 4 — Relations clients-comptes (`lnk` = `TIE_X_CTR`)

### Cas 31 — Combien de liens client-compte sont enregistrés ?

In [ ]:
print(f"Nombre de liens client-compte : {lnk.shape[0]}")

### Cas 32 — Combien de comptes possède chaque client ?

In [ ]:
nb_comptes_par_client = lnk.groupby("NUM_TIE").size().sort_values(ascending=False)
nb_comptes_par_client

### Cas 33 — Quel est le client qui détient le plus grand nombre de comptes ?

In [ ]:
nb_comptes_par_client = lnk.groupby("NUM_TIE").size()
client_top = nb_comptes_par_client.idxmax()
print(f"Client NUM_TIE={client_top} détient {nb_comptes_par_client.max()} comptes")

### Cas 34 — Combien de clients sont mono-compte vs multi-comptes ?

In [ ]:
nb_comptes_par_client = lnk.groupby("NUM_TIE").size()
mono = (nb_comptes_par_client == 1).sum()
multi = (nb_comptes_par_client > 1).sum()
print(f"Clients mono-compte  : {mono}")
print(f"Clients multi-comptes : {multi}")

### Cas 35 — Quelle est la répartition de `FLG_PRE_TTL` (indicateur titulaire principal) ?

In [ ]:
lnk["FLG_PRE_TTL"].value_counts(dropna=False)

### Cas 36 — Quelle est la répartition des codes de rôle `COD_ROL_TTL` (y compris manquants) ?

In [ ]:
lnk["COD_ROL_TTL"].value_counts(dropna=False)

### Cas 37 — Combien de titulaires sont rattachés à chaque compte ?

In [ ]:
nb_titulaires_par_compte = lnk.groupby("IDT_AC").size()
nb_titulaires_par_compte.value_counts().sort_index()

### Cas 38 — Quels comptes ont plusieurs titulaires (cotitularité) ?

In [ ]:
nb_titulaires_par_compte = lnk.groupby("IDT_AC").size()
comptes_cotitulaires = nb_titulaires_par_compte[nb_titulaires_par_compte > 1]
print(f"Comptes avec plusieurs titulaires : {comptes_cotitulaires.shape[0]}")

### Cas 39 — Tous les comptes de `ctr` ont-ils bien un lien dans `lnk` ?

In [ ]:
comptes_sans_lien = ctr[~ctr["IDT_AC"].isin(lnk["IDT_AC"])]
print(f"Comptes CTR sans aucun lien dans TIE_X_CTR : {comptes_sans_lien.shape[0]}")

### Cas 40 — Tous les clients de `tie` ont-ils au moins un compte dans `lnk` ?

In [ ]:
clients_sans_compte = tie[~tie["NUM_TIE"].isin(lnk["NUM_TIE"])]
print(f"Clients de TIE sans aucun compte dans TIE_X_CTR : {clients_sans_compte.shape[0]} sur {tie.shape[0]}")
# -> sur cet extrait pédagogique, la table TIE contient plus de clients que la table de liaison n'en référence :
# rappel utile que deux tables « liées » ne sont pas toujours en parfaite correspondance dans un extrait réel.

## Catégorie 5 — Adresses / géographie (`adr`)

### Cas 41 — Comment se répartissent les clients par pays ?

In [ ]:
adr["COD_PAY_ISO"].value_counts()

### Cas 42 — Quelles sont les 10 villes les plus représentées ?

In [ ]:
adr["NOM_VIL"].value_counts().head(10)

### Cas 43 — Quel est le taux de complétude de l'adresse email ?

In [ ]:
taux_email = adr["ADR_EMA"].notna().mean() * 100
print(f"Taux de complétude de l'email : {taux_email:.1f} %")

### Cas 44 — Quel est le taux de complétude du téléphone mobile vs du téléphone fixe ?

In [ ]:
taux_mobile = adr["NUM_TEL_MOB_INL"].notna().mean() * 100
taux_fixe = adr["NUM_TEL_DOM_INL"].notna().mean() * 100
print(f"Téléphone mobile renseigné : {taux_mobile:.1f} %")
print(f"Téléphone fixe renseigné   : {taux_fixe:.1f} %")

### Cas 45 — Quels clients n'ont aucune coordonnée de contact (ni email, ni téléphone) ?

In [ ]:
sans_contact = adr[
    adr["ADR_EMA"].isna() & adr["NUM_TEL_MOB_INL"].isna() & adr["NUM_TEL_DOM_INL"].isna()
]
print(f"Clients sans aucune coordonnée de contact : {sans_contact.shape[0]}")
sans_contact[["NUM_TIE", "NOM_TIE", "NOM_VIL"]]

### Cas 46 — Quels sont les domaines email les plus fréquents chez les clients ?

In [ ]:
adr["DOMAINE_EMAIL"] = adr["ADR_EMA"].str.split("@").str[-1]
adr["DOMAINE_EMAIL"].value_counts().head(10)

### Cas 47 — Quelle est la répartition des civilités (`LIB_TIT`) ?

In [ ]:
adr["LIB_TIT"].value_counts(dropna=False)

### Cas 48 — Quels clients habitent une ville donnée (ex. Bruxelles) ?

In [ ]:
clients_bruxelles = adr[adr["NOM_VIL"].str.contains("BRUXELLES", case=False, na=False)]
print(f"Clients à Bruxelles : {clients_bruxelles.shape[0]}")
clients_bruxelles[["NUM_TIE", "NOM_TIE", "NOM_VIL"]].head(5)

### Cas 49 — Comment normaliser les noms de ville (espaces, casse) ?

In [ ]:
adr["NOM_VIL_PROPRE"] = adr["NOM_VIL"].str.strip().str.upper()
adr[["NOM_VIL", "NOM_VIL_PROPRE"]].drop_duplicates().head(5)

### Cas 50 — Quelles adresses n'ont pas été mises à jour depuis longtemps (> 3 ans) ?

In [ ]:
aujourdhui = pd.Timestamp.today()
adr["ANCIENNETE_MAJ_JOURS"] = (aujourdhui - adr["DAT_MAJ_ADR"]).dt.days
adresses_anciennes = adr[adr["ANCIENNETE_MAJ_JOURS"] > 3 * 365]
print(f"Adresses non mises à jour depuis plus de 3 ans : {adresses_anciennes.shape[0]}")
adresses_anciennes[["NUM_TIE", "DAT_MAJ_ADR", "ANCIENNETE_MAJ_JOURS"]].sort_values("ANCIENNETE_MAJ_JOURS", ascending=False).head(5)

## Catégorie 6 — Transactions / opérations (`txn`)

### Cas 51 — Combien d'opérations sont enregistrées au total ?

In [ ]:
print(f"Nombre total d'opérations : {txn.shape[0]}")

### Cas 52 — Combien d'opérations par compte ?

In [ ]:
txn.groupby("IDT_AC").size().sort_values(ascending=False).head(10)

### Cas 53 — Quel compte a enregistré le plus grand nombre d'opérations ?

In [ ]:
nb_ope_par_compte = txn.groupby("IDT_AC").size()
compte_top = nb_ope_par_compte.idxmax()
print(f"Compte {compte_top} : {nb_ope_par_compte.max()} opérations")

### Cas 54 — Comment se répartissent les opérations par langue (`COD_LNG_RIU`) ?

In [ ]:
txn["COD_LNG_RIU"].value_counts()

### Cas 55 — Quels sont les 15 types d'opérations les plus fréquents ?

In [ ]:
txn["LIB_OPE_INL_1"].value_counts().head(15)

### Cas 56 — Combien d'opérations correspondent à des virements reçus ?

In [ ]:
virements = txn[txn["LIB_OPE_INL_1"].str.contains("virement de|overschrijving van", case=False, na=False)]
print(f"Virements reçus : {virements.shape[0]}")

### Cas 57 — Combien d'opérations correspondent à des domiciliations ?

In [ ]:
domiciliations = txn[txn["LIB_OPE_INL_1"].str.contains("domicili", case=False, na=False)]
print(f"Domiciliations : {domiciliations.shape[0]}")

### Cas 58 — Combien d'opérations correspondent à des frais bancaires ?

In [ ]:
frais = txn[txn["LIB_OPE_INL_1"].str.contains("frais|kosten", case=False, na=False)]
print(f"Opérations de frais : {frais.shape[0]}")
frais["LIB_OPE_INL_1"].value_counts().head(10)

### Cas 59 — Combien d'opérations par mois ?

In [ ]:
txn.groupby(txn["DAT_CRE_MVT_CPB"].dt.to_period("M")).size()

### Cas 60 — Comment se répartissent les opérations par jour de la semaine ?

In [ ]:
txn["DAT_CRE_MVT_CPB"].dt.day_name().value_counts()

## Catégorie 7 — Croisements clients x comptes (`tie` + `lnk` + `ctr`)

### Cas 61 — Comment construire la table complète client-compte ?

In [ ]:
vue_clients_comptes = tie.merge(lnk, on="NUM_TIE", how="inner").merge(ctr, on="IDT_AC", how="left")
print(vue_clients_comptes.shape)
vue_clients_comptes[["NUM_TIE", "TYPE_CLIENT", "IDT_AC", "STATUT_COMPTE", "SLD_CTR"]].head(5)

### Cas 62 — Quels comptes détient un client donné (ex. NUM_TIE=2900000004654) ?

In [ ]:
num_tie_cible = 2900000004654
comptes_du_client = vue_clients_comptes[vue_clients_comptes["NUM_TIE"] == num_tie_cible]
comptes_du_client[["IDT_AC", "STATUT_COMPTE", "COD_DEV", "SLD_CTR"]]

### Cas 63 — Quel est le solde total détenu par chaque client ?

In [ ]:
solde_total_par_client = vue_clients_comptes.groupby("NUM_TIE")["SLD_CTR"].sum().sort_values(ascending=False)
solde_total_par_client.head(10)

### Cas 64 — Quels sont les 10 clients avec le plus gros patrimoine total (somme des soldes) ?

In [ ]:
vue_clients_comptes.groupby("NUM_TIE")["SLD_CTR"].sum().nlargest(10)

### Cas 65 — Combien de comptes actifs possède chaque client ?

In [ ]:
vue_clients_comptes[vue_clients_comptes["STATUT_COMPTE"] == "Actif"].groupby("NUM_TIE").size().sort_values(ascending=False)

### Cas 66 — Quel type de client (physique/morale) est le plus représenté parmi les comptes en découvert ?

In [ ]:
comptes_decouvert = vue_clients_comptes[vue_clients_comptes["SLD_CTR"] < 0]
comptes_decouvert["TYPE_CLIENT"].value_counts()

### Cas 67 — Quel est l'âge moyen des clients ayant au moins un compte clôturé ?

In [ ]:
clients_avec_compte_cloture = vue_clients_comptes[vue_clients_comptes["STATUT_COMPTE"] == "Clôturé"]
print(f"Âge moyen : {clients_avec_compte_cloture['AGE'].mean():.1f} ans")

### Cas 68 — Comment se croisent la langue de contact et la devise du compte ?

In [ ]:
pd.crosstab(vue_clients_comptes["COD_LNG_CTR"], vue_clients_comptes["COD_DEV"])

### Cas 69 — Quels clients n'ont plus que des comptes clôturés (churn complet) ?

In [ ]:
statut_par_client = vue_clients_comptes.groupby("NUM_TIE")["STATUT_COMPTE"].apply(lambda s: set(s.dropna()))
clients_churn = statut_par_client[statut_par_client == {"Clôturé"}]
print(f"Clients dont tous les comptes sont clôturés : {clients_churn.shape[0]}")
clients_churn

### Cas 70 — Comment segmenter les clients entre « petit portefeuille » et « gros portefeuille » (≥ 3 comptes) ?

In [ ]:
nb_comptes_par_client = vue_clients_comptes.groupby("NUM_TIE")["IDT_AC"].nunique()
segment = np.where(nb_comptes_par_client >= 3, "Gros portefeuille", "Petit portefeuille")
pd.Series(segment, index=nb_comptes_par_client.index).value_counts()

## Catégorie 8 — Croisements comptes x transactions (`ctr` + `txn`)

### Cas 71 — Comment joindre les comptes et leurs opérations ?

In [ ]:
vue_comptes_txn = ctr.merge(txn, on="IDT_AC", how="left", suffixes=("_ctr", "_txn"))
print(vue_comptes_txn.shape)
vue_comptes_txn[["IDT_AC", "STATUT_COMPTE", "DAT_CRE_MVT_CPB", "LIB_OPE_INL_1"]].dropna(subset=["LIB_OPE_INL_1"]).head(5)

### Cas 72 — Combien d'opérations par devise de compte ?

In [ ]:
vue_comptes_txn.dropna(subset=["LIB_OPE_INL_1"]).groupby("COD_DEV").size()

### Cas 73 — Quels comptes actifs n'ont AUCUNE opération enregistrée dans `txn` ?

In [ ]:
# anti-jointure : comptes de ctr dont l'IDT_AC n'apparaît jamais dans txn
comptes_sans_operation = ctr[~ctr["IDT_AC"].isin(txn["IDT_AC"])]
print(f"Comptes sans aucune opération dans l'extrait : {comptes_sans_operation.shape[0]} sur {ctr.shape[0]}")
# -> peu de comptes de CTR se retrouvent dans TXN_X_CTR sur cet extrait : les deux exports
# ne couvrent pas exactement le même périmètre de comptes (cas fréquent avec des exports
# multi-sources non synchronisés).

### Cas 74 — Des comptes clôturés ont-ils encore des opérations après leur date de clôture ?

In [ ]:
vue = ctr.merge(txn, on="IDT_AC", how="inner")
anomalies = vue[(vue["STATUT_COMPTE"] == "Clôturé") & (vue["DAT_CRE_MVT_CPB"] > vue["DAT_CLO_CTR"])]
print(f"Opérations après clôture du compte : {anomalies.shape[0]}")

### Cas 75 — Quelle est la date de la dernière opération enregistrée, par compte ?

In [ ]:
derniere_operation = txn.groupby("IDT_AC")["DAT_CRE_MVT_CPB"].max().sort_values(ascending=False)
derniere_operation.head(10)

### Cas 76 — Quels comptes sont « dormants » (aucune opération depuis plus de 6 mois) ?

In [ ]:
aujourdhui = pd.Timestamp.today()
derniere_operation = txn.groupby("IDT_AC")["DAT_CRE_MVT_CPB"].max()
anciennete_derniere_ope = (aujourdhui - derniere_operation).dt.days
comptes_dormants = anciennete_derniere_ope[anciennete_derniere_ope > 180]
print(f"Comptes sans opération depuis plus de 6 mois : {comptes_dormants.shape[0]}")

### Cas 77 — Quel est le nombre moyen d'opérations par compte, selon le statut du compte ?

In [ ]:
vue_comptes_txn.dropna(subset=["LIB_OPE_INL_1"]).groupby("STATUT_COMPTE").size() / ctr.groupby("STATUT_COMPTE").size()

### Cas 78 — Comment se répartissent les opérations selon le statut du compte ?

In [ ]:
vue_comptes_txn.dropna(subset=["LIB_OPE_INL_1"])["STATUT_COMPTE"].value_counts()

### Cas 79 — Quel mois a connu le plus grand nombre d'opérations, tous comptes confondus ?

In [ ]:
pic_mensuel = txn.groupby(txn["DAT_CRE_MVT_CPB"].dt.to_period("M")).size().sort_values(ascending=False)
print("Mois avec le plus d'opérations :", pic_mensuel.index[0], "->", pic_mensuel.iloc[0], "opérations")

### Cas 80 — Quels comptes ont une seule opération vs beaucoup d'opérations ?

In [ ]:
nb_ope_par_compte = txn.groupby("IDT_AC").size()
print(f"Comptes avec 1 seule opération   : {(nb_ope_par_compte == 1).sum()}")
print(f"Comptes avec plus de 5 opérations : {(nb_ope_par_compte > 5).sum()}")

## Catégorie 9 — KPI et reporting agrégé (tableau de bord)

### Cas 81 — Comment produire un tableau de bord général en une seule fois ?

In [ ]:
tableau_de_bord = pd.Series({
    "Nombre de clients": tie.shape[0],
    "Nombre de comptes": ctr.shape[0],
    "Comptes actifs": (ctr["STATUT_COMPTE"] == "Actif").sum(),
    "Comptes clôturés": (ctr["STATUT_COMPTE"] == "Clôturé").sum(),
    "Nombre d'opérations": txn.shape[0],
    "Solde total connu (EUR)": ctr.loc[ctr["COD_DEV"] == "EUR", "SLD_CTR"].sum(),
})
tableau_de_bord

### Cas 82 — Pivot : nombre de comptes par devise x statut

In [ ]:
ctr.pivot_table(values="IDT_AC", index="COD_DEV", columns="STATUT_COMPTE", aggfunc="count", fill_value=0)

### Cas 83 — Pivot : solde moyen par tranche d'âge du client (titulaire principal)

In [ ]:
vue = tie.merge(lnk, on="NUM_TIE").merge(ctr, on="IDT_AC")
vue.pivot_table(values="SLD_CTR", index="TRANCHE_AGE", aggfunc="mean", observed=False)

### Cas 84 — Pivot : nombre d'opérations par mois x langue

In [ ]:
txn_pivot = txn.copy()
txn_pivot["MOIS"] = txn_pivot["DAT_CRE_MVT_CPB"].dt.to_period("M").astype(str)
txn_pivot.pivot_table(values="IDT_AC", index="MOIS", columns="COD_LNG_RIU", aggfunc="count", fill_value=0).tail(6)

### Cas 85 — Quelle est la part de chaque pays dans le portefeuille clients (%) ?

In [ ]:
(adr["COD_PAY_ISO"].value_counts(normalize=True) * 100).round(1)

### Cas 86 — Quel est le taux de comptes en devise étrangère selon la langue de contact du client ?

In [ ]:
vue = tie.merge(lnk, on="NUM_TIE").merge(ctr, on="IDT_AC")
vue["DEVISE_ETRANGERE"] = vue["COD_DEV"] != "EUR"
vue.groupby("COD_LNG_CTR")["DEVISE_ETRANGERE"].mean().mul(100).round(1)

### Cas 87 — Quel est le pourcentage de comptes en découvert par tranche d'ancienneté ?

In [ ]:
ctr["TRANCHE_ANCIENNETE"] = pd.cut(ctr["ANCIENNETE_ANNEES"], bins=[0, 1, 3, 5, 100], labels=["<1 an", "1-3 ans", "3-5 ans", "5 ans +"])
ctr["EST_DECOUVERT"] = ctr["SLD_CTR"] < 0
(ctr.groupby("TRANCHE_ANCIENNETE", observed=False)["EST_DECOUVERT"].mean() * 100).round(1)

### Cas 88 — Quelles sont les 5 opérations les plus fréquentes, par langue ?

In [ ]:
txn.groupby("COD_LNG_RIU")["LIB_OPE_INL_1"].apply(lambda s: s.value_counts().head(5))

### Cas 89 — Comment exporter le tableau de bord consolidé ?

In [ ]:
resume = ctr.groupby("STATUT_COMPTE").agg(nb_comptes=("IDT_AC", "count"), solde_moyen=("SLD_CTR", "mean"))
resume.to_csv("tableau_de_bord_comptes.csv", sep=";")
print("Export réalisé : tableau_de_bord_comptes.csv")
resume

### Cas 90 — Comment écrire une fonction réutilisable de fiche client ?

In [ ]:
def rapport_client(num_tie):
    """Renvoie un résumé (Series) de la situation d'un client à partir de son NUM_TIE."""
    infos_client = tie[tie["NUM_TIE"] == num_tie]
    if infos_client.empty:
        return pd.Series({"erreur": "client introuvable"})
    comptes_client = lnk[lnk["NUM_TIE"] == num_tie].merge(ctr, on="IDT_AC", how="left")
    return pd.Series({
        "Type de client": infos_client["TYPE_CLIENT"].iloc[0],
        "Langue de contact": infos_client["COD_LNG_CTR"].iloc[0],
        "Nombre de comptes": comptes_client.shape[0],
        "Solde total (EUR)": comptes_client.loc[comptes_client["COD_DEV"] == "EUR", "SLD_CTR"].sum(),
    })

rapport_client(2900000004654)

## Catégorie 10 — Qualité des données / anomalies bancaires

### Cas 91 — Y a-t-il des doublons exacts (lignes strictement identiques) dans chaque table ?

In [ ]:
for nom, df in [("ctr", ctr), ("tie", tie), ("adr", adr), ("lnk", lnk), ("txn", txn)]:
    print(f"{nom} : {df.duplicated().sum()} doublon(s) exact(s)")

### Cas 92 — L'identifiant compte (`IDT_AC`) est-il bien unique dans `CTR` ?

In [ ]:
doublons_idt_ac = ctr["IDT_AC"].duplicated().sum()
print(f"Doublons sur IDT_AC dans CTR : {doublons_idt_ac}")

### Cas 93 — Existe-t-il des comptes clôturés avant leur date d'ouverture (incohérence) ?

In [ ]:
incoherents = ctr[ctr["DAT_CLO_CTR"] < ctr["DAT_OUV_CTR"]]
print(f"Comptes avec date de clôture antérieure à l'ouverture : {incoherents.shape[0]}")

### Cas 94 — Existe-t-il des clients nés après leur date de premier contrat (incohérence) ?

In [ ]:
incoherents = tie[tie["DAT_NAI"] > tie["DAT_PRE_CTR"]]
print(f"Clients nés après leur premier contrat : {incoherents.shape[0]}")

### Cas 95 — Combien de valeurs manquantes par colonne, dans chaque table ?

In [ ]:
for nom, df in [("ctr", ctr), ("tie", tie), ("adr", adr), ("lnk", lnk), ("txn", txn)]:
    print(f"--- {nom} ---")
    print(df.isna().sum()[df.isna().sum() > 0])

### Cas 96 — Quel est le taux de complétude (%) de chaque colonne de `adr`, du plus incomplet au plus complet ?

In [ ]:
taux_completude = (1 - adr.isna().mean()) * 100
taux_completude.sort_values().round(1)

### Cas 97 — Peut-on détecter des soldes statistiquement aberrants (méthode IQR) ?

In [ ]:
q1, q3 = ctr["SLD_CTR"].quantile([0.25, 0.75])
iqr = q3 - q1
borne_basse, borne_haute = q1 - 1.5 * iqr, q3 + 1.5 * iqr
aberrants = ctr[(ctr["SLD_CTR"] < borne_basse) | (ctr["SLD_CTR"] > borne_haute)]
print(f"Bornes IQR : [{borne_basse:.2f} ; {borne_haute:.2f}]")
print(f"Soldes jugés aberrants par la méthode IQR : {aberrants.shape[0]}")
# -> ici plus de 60% des soldes sont à 0 ou manquants, ce qui écrase l'IQR autour de 0 :
# la méthode signale presque tout solde non nul. Bon exemple des LIMITES d'une méthode
# statistique standard sur une donnée très concentrée : il faut croiser avec le métier.

### Cas 98 — Certains comptes utilisent-ils une devise hors de la liste autorisée par la banque ?

In [ ]:
devises_autorisees = ["EUR", "USD", "GBP", "CHF"]   # exemple pédagogique de politique interne
devises_a_verifier = ctr[~ctr["COD_DEV"].isin(devises_autorisees)]
print(f"Comptes en devise hors politique interne : {devises_a_verifier.shape[0]}")
devises_a_verifier["COD_DEV"].value_counts()

### Cas 99 — Combien de codes postaux ne respectent pas le format belge à 4 chiffres ?

In [ ]:
codes_postaux = adr["COD_PST"].dropna().astype(int).astype(str)
hors_format = codes_postaux[codes_postaux.str.len() != 4]
print(f"Codes postaux hors format 4 chiffres : {hors_format.shape[0]}")

### Cas 100 — Comment produire un rapport de qualité de données consolidé, multi-tables ?

In [ ]:
def rapport_qualite():
    """Construit un tableau de synthèse qualité (lignes, doublons, cellules manquantes) pour chaque table."""
    tables = {"CTR": ctr, "TIE": tie, "TIE_ADR": adr, "TIE_X_CTR": lnk, "TXN_X_CTR": txn}
    lignes = []
    for nom, df in tables.items():
        lignes.append({
            "table": nom,
            "nb_lignes": df.shape[0],
            "nb_colonnes": df.shape[1],
            "doublons": df.duplicated().sum(),
            "cellules_manquantes": df.isna().sum().sum(),
            "taux_completude_pct": round((1 - df.isna().mean().mean()) * 100, 1),
        })
    return pd.DataFrame(lignes)

rapport_qualite()

<a id="part3"></a>
# Partie 3 — Fiche récapitulative des fonctions Pandas

Toutes les fonctions et méthodes utilisées dans ce notebook, classées par thème.

## Chargement / export

| Fonction | Rôle |
|---|---|
| `pd.read_csv(chemin, sep=";", encoding="cp1252")` | Charge un fichier CSV dans un DataFrame |
| `df.to_csv(chemin, sep=";")` | Écrit un DataFrame dans un fichier CSV |
| `df.to_excel(chemin)` | Écrit un DataFrame dans un fichier Excel (nécessite `openpyxl`) |

## Exploration

| Fonction | Rôle |
|---|---|
| `df.head(n)` / `df.tail(n)` | n premières / dernières lignes |
| `df.sample(n)` | n lignes prises au hasard |
| `df.shape` | (nombre de lignes, nombre de colonnes) |
| `df.info()` | Résumé : colonnes, types, mémoire |
| `df.describe()` | Statistiques des colonnes numériques |
| `df.dtypes` | Type de chaque colonne |
| `df.columns` / `df.index` | Noms des colonnes / étiquettes des lignes |

## Sélection

| Fonction | Rôle |
|---|---|
| `df["col"]` | Sélectionne une colonne (Series) |
| `df[["col1","col2"]]` | Sélectionne plusieurs colonnes (DataFrame) |
| `df.loc[ligne, colonne]` | Sélection par étiquette |
| `df.iloc[position]` | Sélection par position numérique |
| `df.select_dtypes(include=...)` | Sélectionne les colonnes d'un type donné |
| `df.query("condition")` | Filtre avec une syntaxe en texte |

## Filtrage

| Fonction | Rôle |
|---|---|
| `df[condition]` | Filtre les lignes selon une condition booléenne |
| `&`, `\|`, `~` | ET, OU, NON logiques (entre parenthèses) |
| `df["col"].isin([...])` | Teste l'appartenance à une liste |
| `df["col"].between(a, b)` | Teste l'appartenance à un intervalle |

## Valeurs manquantes

| Fonction | Rôle |
|---|---|
| `df.replace(valeur, pd.NA)` | Remplace une valeur par une vraie valeur manquante |
| `df.isna()` / `df.notna()` | Détecte les valeurs manquantes / renseignées |
| `df.dropna(subset=[...])` | Supprime les lignes avec valeurs manquantes |
| `df.fillna(valeur)` | Remplace les valeurs manquantes |

## Types

| Fonction | Rôle |
|---|---|
| `pd.to_numeric(col, errors="coerce")` | Convertit en nombre |
| `pd.to_datetime(col, format=..., errors="coerce")` | Convertit en date |
| `df["col"].astype(type)` | Convertit vers un type précis |
| `df["col"].map({...})` | Remplace des codes par des libellés |

## Tri et top/bottom

| Fonction | Rôle |
|---|---|
| `df.sort_values("col", ascending=False)` | Trie selon une colonne |
| `df.sort_index()` | Trie selon l'index |
| `df.nlargest(n, "col")` / `df.nsmallest(n, "col")` | n plus grandes / petites valeurs |

## Création de colonnes

| Fonction | Rôle |
|---|---|
| `df["nouvelle"] = ...` | Opération vectorisée directe |
| `np.where(condition, a, b)` | SI/ALORS/SINON vectorisé |
| `df["col"].apply(fonction)` | Applique une fonction personnalisée |
| `df.assign(nouvelle=...)` | Crée une colonne sans modifier le DataFrame d'origine |
| `pd.cut(col, bins=..., labels=...)` | Découpe une variable numérique en tranches |

## Texte (`.str`)

| Fonction | Rôle |
|---|---|
| `.str.upper()` / `.str.lower()` | Majuscules / minuscules |
| `.str.strip()` | Supprime les espaces en début/fin |
| `.str.contains("motif", case=False, na=False)` | Teste la présence d'un motif |
| `.str.split("sep")` | Découpe une chaîne de caractères |
| `.str.len()` | Longueur du texte |

## Dates (`.dt`)

| Fonction | Rôle |
|---|---|
| `.dt.year` / `.dt.month` / `.dt.day` | Extrait année / mois / jour |
| `.dt.day_name()` | Nom du jour de la semaine |
| `.dt.to_period("M")` | Regroupe par mois |
| `(date1 - date2).dt.days` | Différence entre deux dates, en jours |

## Doublons

| Fonction | Rôle |
|---|---|
| `df.duplicated()` | Détecte les lignes en double |
| `df.drop_duplicates()` | Supprime les doublons |

## Comptages

| Fonction | Rôle |
|---|---|
| `df["col"].unique()` | Valeurs distinctes |
| `df["col"].nunique()` | Nombre de valeurs distinctes |
| `df["col"].value_counts(normalize=True)` | Comptage (ou %) par valeur |

## Agrégation (GroupBy)

| Fonction | Rôle |
|---|---|
| `df.groupby("col").size()` | Nombre de lignes par groupe |
| `df.groupby("col").agg({...})` | Plusieurs statistiques par groupe |
| `df.groupby("col")["x"].transform("mean")` | Statistique de groupe reportée sur chaque ligne |

## Tableaux croisés

| Fonction | Rôle |
|---|---|
| `pd.crosstab(a, b)` | Compte les combinaisons entre deux colonnes |
| `df.pivot_table(values=, index=, columns=, aggfunc=)` | Tableau croisé dynamique |

## Fusion / réorganisation

| Fonction | Rôle |
|---|---|
| `df1.merge(df2, on="clé", how="left")` | Jointure entre deux tables |
| `pd.concat([df1, df2])` | Empile deux DataFrames |
| `df.melt(id_vars=, var_name=, value_name=)` | Format large -> format long |
| `df.set_index("col")` / `df.reset_index()` | Change / réinitialise l'index |

## Cumuls et fenêtres

| Fonction | Rôle |
|---|---|
| `df["col"].cumsum()` | Somme cumulée |
| `df["col"].rolling(n).mean()` | Moyenne glissante sur n périodes |

## Bonnes pratiques

| Fonction | Rôle |
|---|---|
| `df.copy()` | Copie indépendante (évite `SettingWithCopyWarning`) |
| opération vectorisée plutôt qu'une boucle `for` | Bien plus rapide sur de gros volumes |

---

## Pour aller plus loin

- [Documentation officielle Pandas](https://pandas.pydata.org/docs/)
- [Pandas — 10 minutes to pandas](https://pandas.pydata.org/docs/user_guide/10min.html)
- [W3Schools — Pandas Tutorial](https://www.w3schools.com/python/pandas/default.asp)

*Support réalisé pour la formation Beobank · Orsys — Python / Pandas appliqué aux données bancaires.*